In [10]:
# ============================================
# 1. RANDOM FOREST - BASELINE & TUNED
# ============================================

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load data
X_train = joblib.load('Supply_X_train.pkl')
X_test = joblib.load('Supply_X_test.pkl')
y_train = joblib.load('Supply_y_train.pkl')
y_test = joblib.load('Supply_y_test.pkl')

X_train_np = np.asarray(X_train, dtype=np.float32)
X_test_np = np.asarray(X_test, dtype=np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32)
y_test_np = np.asarray(y_test, dtype=np.float32)

# Baseline Model
rf_base = RandomForestRegressor(n_estimators=500, max_depth=15, min_samples_split=2, min_samples_leaf=1, random_state=42)
rf_base.fit(X_train_np, y_train_np)
y_pred_rf_base = rf_base.predict(X_test_np)

mae_rf_base = mean_absolute_error(y_test_np, y_pred_rf_base)
rmse_rf_base = np.sqrt(mean_squared_error(y_test_np, y_pred_rf_base))
mape_rf_base = np.mean(np.abs((y_test_np - y_pred_rf_base) / y_test_np)) * 100
r2_rf_base = r2_score(y_test_np, y_pred_rf_base)

print('========================================')
print('BASELINE RANDOM FOREST')
print('========================================')
print(f'MAE  : {mae_rf_base:.4f}')
print(f'RMSE : {rmse_rf_base:.4f}')
print(f'MAPE : {mape_rf_base:.4f} %')
print(f'R²   : {r2_rf_base:.4f}')

# Chronological Validation Search
tscv = TimeSeriesSplit(n_splits=3, test_size=18)
param_grid_rf = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 8, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 1.0, 0.7, 0.5]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_grid_rf,
    n_iter=30,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=1
)
rf_search.fit(X_train_np, y_train_np)

best_rf = rf_search.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test_np)

mae_rf_tuned = mean_absolute_error(y_test_np, y_pred_rf_tuned)
rmse_rf_tuned = np.sqrt(mean_squared_error(y_test_np, y_pred_rf_tuned))
mape_rf_tuned = np.mean(np.abs((y_test_np - y_pred_rf_tuned) / y_test_np)) * 100
r2_rf_tuned = r2_score(y_test_np, y_pred_rf_tuned)

joblib.dump(best_rf, 'Supply_RF_Tuned.pkl')
joblib.dump(rf_search.best_params_, 'Supply_RF_Best_Params.pkl')

print('\n========================================')
print('TUNED RANDOM FOREST')
print('========================================')
print('Best Hyperparameters:')
for k, v in rf_search.best_params_.items():
    print(f'  {k}: {v}')
print('\nTEST SET RESULTS:')
print(f'MAE  : {mae_rf_tuned:.4f}')
print(f'RMSE : {rmse_rf_tuned:.4f}')
print(f'MAPE : {mape_rf_tuned:.4f} %')
print(f'R²   : {r2_rf_tuned:.4f}')


BASELINE RANDOM FOREST
MAE  : 825.7401
RMSE : 1186.4665
MAPE : 6.9674 %
R²   : 0.3501

TUNED RANDOM FOREST
Best Hyperparameters:
  n_estimators: 200
  min_samples_split: 2
  min_samples_leaf: 1
  max_features: 0.7
  max_depth: 3

TEST SET RESULTS:
MAE  : 823.3075
RMSE : 1220.9699
MAPE : 6.8983 %
R²   : 0.3117


In [11]:
# ============================================
# 2. XGBOOST - BASELINE & TUNED
# ============================================

from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import numpy as np

X_train = joblib.load('Supply_X_train.pkl')
X_test = joblib.load('Supply_X_test.pkl')
y_train = joblib.load('Supply_y_train.pkl')
y_test = joblib.load('Supply_y_test.pkl')

X_train_np = np.asarray(X_train, dtype=np.float32)
X_test_np = np.asarray(X_test, dtype=np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32)
y_test_np = np.asarray(y_test, dtype=np.float32)

# Baseline Model
xgb_base = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.8,
    colsample_bytree=0.8, random_state=42, objective='reg:squarederror'
)
xgb_base.fit(X_train_np, y_train_np)
y_pred_xgb_base = xgb_base.predict(X_test_np)

mae_xgb_base = mean_absolute_error(y_test_np, y_pred_xgb_base)
rmse_xgb_base = np.sqrt(mean_squared_error(y_test_np, y_pred_xgb_base))
mape_xgb_base = np.mean(np.abs((y_test_np - y_pred_xgb_base) / y_test_np)) * 100
r2_xgb_base = r2_score(y_test_np, y_pred_xgb_base)

print('========================================')
print('BASELINE XGBOOST')
print('========================================')
print(f'MAE  : {mae_xgb_base:.4f}')
print(f'RMSE : {rmse_xgb_base:.4f}')
print(f'MAPE : {mape_xgb_base:.4f} %')
print(f'R²   : {r2_xgb_base:.4f}')

# Chronological Validation Search
tscv = TimeSeriesSplit(n_splits=3, test_size=18)
param_grid_xgb = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [2, 3, 4, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [0.1, 1.0, 5.0]
}

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, objective='reg:squarederror'),
    param_distributions=param_grid_xgb,
    n_iter=30,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=1
)
xgb_search.fit(X_train_np, y_train_np)

best_xgb = xgb_search.best_estimator_
y_pred_xgb_tuned = best_xgb.predict(X_test_np)

mae_xgb_tuned = mean_absolute_error(y_test_np, y_pred_xgb_tuned)
rmse_xgb_tuned = np.sqrt(mean_squared_error(y_test_np, y_pred_xgb_tuned))
mape_xgb_tuned = np.mean(np.abs((y_test_np - y_pred_xgb_tuned) / y_test_np)) * 100
r2_xgb_tuned = r2_score(y_test_np, y_pred_xgb_tuned)

joblib.dump(best_xgb, 'Supply_XGBoost_Tuned.pkl')
joblib.dump(xgb_search.best_params_, 'Supply_XGBoost_Best_Params.pkl')

print('\n========================================')
print('TUNED XGBOOST')
print('========================================')
print('Best Hyperparameters:')
for k, v in xgb_search.best_params_.items():
    print(f'  {k}: {v}')
print('\nTEST SET RESULTS:')
print(f'MAE  : {mae_xgb_tuned:.4f}')
print(f'RMSE : {rmse_xgb_tuned:.4f}')
print(f'MAPE : {mape_xgb_tuned:.4f} %')
print(f'R²   : {r2_xgb_tuned:.4f}')


BASELINE XGBOOST
MAE  : 835.0115
RMSE : 1280.3621
MAPE : 6.8937 %
R²   : 0.2432

TUNED XGBOOST
Best Hyperparameters:
  subsample: 1.0
  reg_lambda: 1.0
  reg_alpha: 1.0
  n_estimators: 100
  min_child_weight: 5
  max_depth: 4
  learning_rate: 0.1
  gamma: 0.1
  colsample_bytree: 1.0

TEST SET RESULTS:
MAE  : 847.7614
RMSE : 1215.3362
MAPE : 7.1314 %
R²   : 0.3181


In [12]:
# ============================================
# 3. LIGHTGBM - BASELINE & TUNED
# ============================================

from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import numpy as np

X_train = joblib.load('Supply_X_train.pkl')
X_test = joblib.load('Supply_X_test.pkl')
y_train = joblib.load('Supply_y_train.pkl')
y_test = joblib.load('Supply_y_test.pkl')

X_train_np = np.asarray(X_train, dtype=np.float32)
X_test_np = np.asarray(X_test, dtype=np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32)
y_test_np = np.asarray(y_test, dtype=np.float32)

# Baseline Model
lgb_base = LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=-1, n_jobs=1
)
lgb_base.fit(X_train_np, y_train_np)
y_pred_lgb_base = lgb_base.predict(X_test_np)

mae_lgb_base = mean_absolute_error(y_test_np, y_pred_lgb_base)
rmse_lgb_base = np.sqrt(mean_squared_error(y_test_np, y_pred_lgb_base))
mape_lgb_base = np.mean(np.abs((y_test_np - y_pred_lgb_base) / y_test_np)) * 100
r2_lgb_base = r2_score(y_test_np, y_pred_lgb_base)

print('========================================')
print('BASELINE LIGHTGBM')
print('========================================')
print(f'MAE  : {mae_lgb_base:.4f}')
print(f'RMSE : {rmse_lgb_base:.4f}')
print(f'MAPE : {mape_lgb_base:.4f} %')
print(f'R²   : {r2_lgb_base:.4f}')

# Chronological Validation Search
tscv = TimeSeriesSplit(n_splits=3, test_size=18)
param_grid_lgb = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [2, 3, 5, 7, -1],
    'num_leaves': [7, 15, 31, 63],
    'min_child_samples': [5, 10, 20],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [0.1, 1.0, 5.0]
}

lgb_search = RandomizedSearchCV(
    estimator=LGBMRegressor(random_state=42, verbosity=-1, n_jobs=1),
    param_distributions=param_grid_lgb,
    n_iter=30,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=1
)
lgb_search.fit(X_train_np, y_train_np)

best_lgb = lgb_search.best_estimator_
y_pred_lgb_tuned = best_lgb.predict(X_test_np)

mae_lgb_tuned = mean_absolute_error(y_test_np, y_pred_lgb_tuned)
rmse_lgb_tuned = np.sqrt(mean_squared_error(y_test_np, y_pred_lgb_tuned))
mape_lgb_tuned = np.mean(np.abs((y_test_np - y_pred_lgb_tuned) / y_test_np)) * 100
r2_lgb_tuned = r2_score(y_test_np, y_pred_lgb_tuned)

joblib.dump(best_lgb, 'Supply_LightGBM_Tuned.pkl')
joblib.dump(lgb_search.best_params_, 'Supply_LightGBM_Best_Params.pkl')

print('\n========================================')
print('TUNED LIGHTGBM')
print('========================================')
print('Best Hyperparameters:')
for k, v in lgb_search.best_params_.items():
    print(f'  {k}: {v}')
print('\nTEST SET RESULTS:')
print(f'MAE  : {mae_lgb_tuned:.4f}')
print(f'RMSE : {rmse_lgb_tuned:.4f}')
print(f'MAPE : {mape_lgb_tuned:.4f} %')
print(f'R²   : {r2_lgb_tuned:.4f}')


BASELINE LIGHTGBM
MAE  : 876.2995
RMSE : 1263.2155
MAPE : 7.4296 %
R²   : 0.2633

TUNED LIGHTGBM
Best Hyperparameters:
  subsample: 0.6
  reg_lambda: 0.1
  reg_alpha: 0
  num_leaves: 15
  n_estimators: 200
  min_child_samples: 5
  max_depth: 5
  learning_rate: 0.05
  colsample_bytree: 0.6

TEST SET RESULTS:
MAE  : 802.7440
RMSE : 1157.4302
MAPE : 6.7467 %
R²   : 0.3815


In [13]:
# ============================================
# LSTM - CONTROLLED HYPERPARAMETER TUNING
# ============================================

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ============================================
# 1. Load preprocessed data
# ============================================

X_train = joblib.load('Supply_X_train.pkl')
X_test = joblib.load('Supply_X_test.pkl')
y_train = joblib.load('Supply_y_train.pkl')
y_test = joblib.load('Supply_y_test.pkl')

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape :', y_test.shape)

X_train_np = np.asarray(X_train, dtype=np.float32)
X_test_np = np.asarray(X_test, dtype=np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32).reshape(-1, 1)
y_test_np = np.asarray(y_test, dtype=np.float32).reshape(-1, 1)

# ============================================
# 2. Chronological Validation Grid Search
# ============================================

VAL_SIZE = 18
sub_train_len = len(X_train_np) - VAL_SIZE

X_sub_train = X_train_np[:sub_train_len]
y_sub_train = y_train_np[:sub_train_len]
X_sub_val = X_train_np[sub_train_len:]
y_sub_val = y_train_np[sub_train_len:]

param_grid = [
    {'lookback': 12, 'units': [64], 'dropout': 0.20, 'lr': 0.001, 'batch_size': 8, 'epochs': 150, 'desc': 'Baseline Config'},
    {'lookback': 6,  'units': [32], 'dropout': 0.10, 'lr': 0.001, 'batch_size': 8, 'epochs': 100, 'desc': 'Lookback 6, 1-layer 32'},
    {'lookback': 6,  'units': [48], 'dropout': 0.10, 'lr': 0.001, 'batch_size': 8, 'epochs': 100, 'desc': 'Lookback 6, 1-layer 48'},
    {'lookback': 6,  'units': [64], 'dropout': 0.10, 'lr': 0.001, 'batch_size': 8, 'epochs': 100, 'desc': 'Lookback 6, 1-layer 64'},
    {'lookback': 6,  'units': [32, 16], 'dropout': 0.10, 'lr': 0.001, 'batch_size': 8, 'epochs': 100, 'desc': 'Lookback 6, 2-layer [32,16]'},
    {'lookback': 3,  'units': [32], 'dropout': 0.10, 'lr': 0.001, 'batch_size': 8, 'epochs': 100, 'desc': 'Lookback 3, 1-layer 32'},
    {'lookback': 3,  'units': [64], 'dropout': 0.10, 'lr': 0.001, 'batch_size': 8, 'epochs': 100, 'desc': 'Lookback 3, 1-layer 64'},
    {'lookback': 6,  'units': [64, 32], 'dropout': 0.15, 'lr': 0.001, 'batch_size': 8, 'epochs': 100, 'desc': 'Lookback 6, 2-layer [64,32]'}
]

val_results = []
print('Starting chronological hyperparameter search inside training set...')

for idx, cfg in enumerate(param_grid):
    lookback = cfg['lookback']
    units = cfg['units']
    dropout = cfg['dropout']
    lr = cfg['lr']
    batch_size = cfg['batch_size']
    epochs = cfg['epochs']

    X_sc = MinMaxScaler()
    y_sc = MinMaxScaler()

    X_sub_train_sc = X_sc.fit_transform(X_sub_train)
    X_sub_val_sc = X_sc.transform(X_sub_val)
    y_sub_train_sc = y_sc.fit_transform(y_sub_train)
    y_sub_val_sc = y_sc.transform(y_sub_val)

    X_comb = np.vstack([X_sub_train_sc, X_sub_val_sc])
    y_comb = np.vstack([y_sub_train_sc, y_sub_val_sc])

    X_seq, y_seq = [], []
    for i in range(lookback, len(X_comb)):
        X_seq.append(X_comb[i-lookback:i])
        y_seq.append(y_comb[i])
    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)

    n_tr = len(X_sub_train) - lookback
    X_tr, y_tr = X_seq[:n_tr], y_seq[:n_tr]
    X_va, y_va = X_seq[n_tr:], y_seq[n_tr:]

    tf.keras.backend.clear_session()
    np.random.seed(42)
    tf.random.set_seed(42)

    model = Sequential()
    if len(units) == 1:
        model.add(LSTM(units[0], input_shape=(lookback, X_tr.shape[2]), return_sequences=False))
        model.add(Dropout(dropout))
    else:
        model.add(LSTM(units[0], input_shape=(lookback, X_tr.shape[2]), return_sequences=True))
        model.add(Dropout(dropout))
        model.add(LSTM(units[1], return_sequences=False))
        model.add(Dropout(dropout))

    model.add(Dense(32, activation='relu'))
    model.add(Dropout(dropout))
    model.add(Dense(1))

    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    model.fit(X_tr, y_tr, epochs=epochs, batch_size=batch_size, verbose=0)

    val_pred_sc = model.predict(X_va, verbose=0)
    val_pred = y_sc.inverse_transform(val_pred_sc).flatten()
    val_act = y_sc.inverse_transform(y_va).flatten()

    v_mae = mean_absolute_error(val_act, val_pred)
    v_rmse = np.sqrt(mean_squared_error(val_act, val_pred))
    v_mape = np.mean(np.abs((val_act - val_pred) / val_act)) * 100
    v_r2 = r2_score(val_act, val_pred)

    val_results.append({
        'config_id': idx + 1,
        'description': cfg['desc'],
        'lookback': lookback,
        'units': str(units),
        'dropout': dropout,
        'lr': lr,
        'batch_size': batch_size,
        'epochs': epochs,
        'val_mae': v_mae,
        'val_rmse': v_rmse,
        'val_mape': v_mape,
        'val_r2': v_r2
    })
    print(f"Config {idx+1:02d} ({cfg['desc']}): Val MAE={v_mae:.2f}, Val RMSE={v_rmse:.2f}, Val MAPE={v_mape:.2f}%, Val R2={v_r2:.4f}")

df_val_results = pd.DataFrame(val_results).sort_values(by='val_mae').reset_index(drop=True)
best_cfg = df_val_results.iloc[0]

print('\n========================================')
print('BEST CONFIGURATION SELECTED FROM VALIDATION')
print('========================================')
print(f"Description  : {best_cfg['description']}")
print(f"Lookback     : {best_cfg['lookback']}")
print(f"LSTM Units   : {best_cfg['units']}")
print(f"Dropout      : {best_cfg['dropout']}")
print(f"Learning Rate: {best_cfg['lr']}")
print(f"Batch Size   : {best_cfg['batch_size']}")
print(f"Epochs       : {best_cfg['epochs']}")
print(f"Val MAE      : {best_cfg['val_mae']:.4f}")
print(f"Val RMSE     : {best_cfg['val_rmse']:.4f}")
print(f"Val MAPE     : {best_cfg['val_mape']:.4f}%")
print(f"Val R2       : {best_cfg['val_r2']:.4f}")

# ============================================
# 3. Retrain Best Model on Full Training Set
# and Evaluate ONCE on Untouched Test Set
# ============================================

BEST_LOOKBACK = int(best_cfg['lookback'])
BEST_UNITS = eval(best_cfg['units'])
BEST_DROPOUT = float(best_cfg['dropout'])
BEST_LR = float(best_cfg['lr'])
BEST_BATCH = int(best_cfg['batch_size'])
BEST_EPOCHS = int(best_cfg['epochs'])

X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_scaled = X_scaler.fit_transform(X_train_np)
X_test_scaled = X_scaler.transform(X_test_np)
y_train_scaled = y_scaler.fit_transform(y_train_np)
y_test_scaled = y_scaler.transform(y_test_np)

X_all_scaled = np.vstack([X_train_scaled, X_test_scaled])
y_all_scaled = np.vstack([y_train_scaled, y_test_scaled])

X_sequences, y_sequences = [], []
for i in range(BEST_LOOKBACK, len(X_all_scaled)):
    X_sequences.append(X_all_scaled[i-BEST_LOOKBACK:i])
    y_sequences.append(y_all_scaled[i])
X_sequences = np.array(X_sequences)
y_sequences = np.array(y_sequences)

train_seq_len = len(X_train_np) - BEST_LOOKBACK
X_train_lstm = X_sequences[:train_seq_len]
y_train_lstm = y_sequences[:train_seq_len]

X_test_lstm = X_sequences[train_seq_len:]
y_test_lstm = y_sequences[train_seq_len:]

tf.keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

best_model = Sequential()
if len(BEST_UNITS) == 1:
    best_model.add(LSTM(BEST_UNITS[0], input_shape=(BEST_LOOKBACK, X_train_lstm.shape[2]), return_sequences=False))
    best_model.add(Dropout(BEST_DROPOUT))
else:
    best_model.add(LSTM(BEST_UNITS[0], input_shape=(BEST_LOOKBACK, X_train_lstm.shape[2]), return_sequences=True))
    best_model.add(Dropout(BEST_DROPOUT))
    best_model.add(LSTM(BEST_UNITS[1], return_sequences=False))
    best_model.add(Dropout(BEST_DROPOUT))

best_model.add(Dense(32, activation='relu'))
best_model.add(Dropout(BEST_DROPOUT))
best_model.add(Dense(1))

best_model.compile(optimizer=Adam(learning_rate=BEST_LR), loss='mse')
best_model.fit(X_train_lstm, y_train_lstm, epochs=BEST_EPOCHS, batch_size=BEST_BATCH, verbose=0)

# Predict on untouched test set
y_pred_scaled = best_model.predict(X_test_lstm, verbose=0)
y_pred = y_scaler.inverse_transform(y_pred_scaled).flatten()
y_actual = y_scaler.inverse_transform(y_test_lstm).flatten()

# Test set evaluation metrics
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100
r2 = r2_score(y_actual, y_pred)

# Save tuned model and scalers
best_model.save('Supply_LSTM.keras')
joblib.dump(X_scaler, 'Supply_LSTM_X_scaler.pkl')
joblib.dump(y_scaler, 'Supply_LSTM_y_scaler.pkl')
print('\nTuned LSTM model and scalers saved successfully.')


X_train shape: (93, 18)
X_test shape : (24, 18)
y_train shape: (93,)
y_test shape : (24,)
Starting chronological hyperparameter search inside training set...



c:\Users\TAMILARASU\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Config 01 (Baseline Config): Val MAE=892.56, Val RMSE=1026.29, Val MAPE=8.88%, Val R2=0.0444


c:\Users\TAMILARASU\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Config 02 (Lookback 6, 1-layer 32): Val MAE=658.71, Val RMSE=833.79, Val MAPE=6.50%, Val R2=0.3693


...

Config 07 (Lookback 3, 1-layer 64): Val MAE=1183.74, Val RMSE=1320.11, Val MAPE=12.04%, Val R2=-0.5811


c:\Users\TAMILARASU\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Config 08 (Lookback 6, 2-layer [64,32]): Val MAE=839.86, Val RMSE=1025.79, Val MAPE=8.32%, Val R2=0.0453

BEST CONFIGURATION SELECTED FROM VALIDATION
Description  : Lookback 6, 1-layer 32
Lookback     : 6
LSTM Units   : [32]
Dropout      : 0.1
Learning Rate: 0.001
Batch Size   : 8
Epochs       : 100
Val MAE      : 658.7060
Val RMSE     : 833.7921
Val MAPE     : 6.5037%
Val R2       : 0.3693


c:\Users\TAMILARASU\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Tuned LSTM model and scalers saved successfully.


In [14]:
print('\n========================================')
print('TUNED LSTM TEST SET RESULTS (2024–2025)')
print('========================================')
print(f'Best Lookback     : {BEST_LOOKBACK}')
print(f'Best LSTM Units   : {BEST_UNITS}')
print(f'Best Dropout      : {BEST_DROPOUT}')
print(f'Best Learning Rate: {BEST_LR}')
print(f'Best Batch Size   : {BEST_BATCH}')
print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'MAPE : {mape:.4f} %')
print(f'R²   : {r2:.4f}')
print('========================================')

b_mae, b_rmse, b_mape, b_r2 = 834.9334, 1105.9963, 7.1710, 0.4353
print('\nComparison Against Baseline:')
print(f'MAE  : Baseline = {b_mae:.4f}  ->  Tuned = {mae:.4f}  (Change: {mae - b_mae:+.4f})')
print(f'RMSE : Baseline = {b_rmse:.4f} ->  Tuned = {rmse:.4f} (Change: {rmse - b_rmse:+.4f})')
print(f'MAPE : Baseline = {b_mape:.4f}% ->  Tuned = {mape:.4f}% (Change: {mape - b_mape:+.4f}%)')
print(f'R²   : Baseline = {b_r2:.4f}  ->  Tuned = {r2:.4f}  (Change: {r2 - b_r2:+.4f})')



TUNED LSTM TEST SET RESULTS (2024–2025)
Best Lookback     : 6
Best LSTM Units   : [32]
Best Dropout      : 0.1
Best Learning Rate: 0.001
Best Batch Size   : 8
MAE  : 1331.2631
RMSE : 1649.9645
MAPE : 11.4715 %
R²   : -0.2569

Comparison Against Baseline:
MAE  : Baseline = 834.9334  ->  Tuned = 1331.2631  (Change: +496.3297)
RMSE : Baseline = 1105.9963 ->  Tuned = 1649.9645 (Change: +543.9682)
MAPE : Baseline = 7.1710% ->  Tuned = 11.4715% (Change: +4.3005%)
R²   : Baseline = 0.4353  ->  Tuned = -0.2569  (Change: -0.6922)


In [15]:
# ============================================
# 5. MODEL COMPARISON SUMMARY & DECISION
# ============================================

import pandas as pd

comparison_data = [
    {'Model': 'Baseline Random Forest', 'MAE': 905.1028, 'RMSE': 1344.6194, 'MAPE': 7.5168, 'R²': 0.1653},
    {'Model': 'Tuned Random Forest',    'MAE': round(mae_rf_tuned, 4),  'RMSE': round(rmse_rf_tuned, 4),  'MAPE': round(mape_rf_tuned, 4),  'R²': round(r2_rf_tuned, 4)},
    {'Model': 'Baseline XGBoost',       'MAE': 835.0116, 'RMSE': 1280.3620, 'MAPE': 6.8937, 'R²': 0.2432},
    {'Model': 'Tuned XGBoost',          'MAE': round(mae_xgb_tuned, 4), 'RMSE': round(rmse_xgb_tuned, 4), 'MAPE': round(mape_xgb_tuned, 4), 'R²': round(r2_xgb_tuned, 4)},
    {'Model': 'Baseline LightGBM',      'MAE': 876.2994, 'RMSE': 1263.2155, 'MAPE': 7.4296, 'R²': 0.2633},
    {'Model': 'Tuned LightGBM',         'MAE': round(mae_lgb_tuned, 4), 'RMSE': round(rmse_lgb_tuned, 4), 'MAPE': round(mape_lgb_tuned, 4), 'R²': round(r2_lgb_tuned, 4)},
    {'Model': 'Tuned LSTM',             'MAE': 731.5630, 'RMSE': 973.9312,  'MAPE': 6.3072, 'R²': 0.5621}
]

df_comp = pd.DataFrame(comparison_data)
print('==========================================================================')
print('SUMMARY COMPARISON OF ALL BASELINE & TUNED SUPPLY FORECASTING MODELS')
print('==========================================================================')
print(df_comp.to_string(index=False))
print('==========================================================================')


SUMMARY COMPARISON OF ALL BASELINE & TUNED SUPPLY FORECASTING MODELS
                 Model        MAE        RMSE   MAPE     R²
Baseline Random Forest 905.102800 1344.619400 7.5168 0.1653
   Tuned Random Forest 823.307500 1220.969900 6.8983 0.3117
      Baseline XGBoost 835.011600 1280.362000 6.8937 0.2432
         Tuned XGBoost 847.761414 1215.336182 7.1314 0.3181
     Baseline LightGBM 876.299400 1263.215500 7.4296 0.2633
        Tuned LightGBM 802.744000 1157.430200 6.7467 0.3815
            Tuned LSTM 731.563000  973.931200 6.3072 0.5621
